In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from catboost import CatBoostClassifier

# ====================== 配置区域 ======================
TRAIN_PATH = r"C:\Users\23849\Desktop\Predicting Irrigation Need\train.csv"
TEST_PATH = r"C:\Users\23849\Desktop\Predicting Irrigation Need\test.csv"
SUBMISSION_OUTPUT_PATH = r"C:\Users\23849\Desktop\Predicting Irrigation Need\submission_final.csv"

# 训练模式选择：'CPU' 或 'GPU'
# 注意：如果选'GPU'但报错，请改回'CPU'
TRAINING_MODE = 'GPU' 

# ====================== 1. 数据读取 ======================
print("正在读取数据...")
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)
submission = df_test[['id']]

# 分离标签和特征
y = df_train['Irrigation_Need'].map({'Low': 0, 'Medium': 1, 'High': 2})
X = df_train.drop(['id', 'Irrigation_Need'], axis=1)
X_test = df_test.drop(['id'], axis=1)

# ====================== 2. 特征工程 ======================
print("正在构造特征...")
def feature_engineering(df):
    df = df.copy()
    df['ET0'] = (df['Temperature_C'] * df['Wind_Speed_kmh'] * df['Sunlight_Hours']) / (df['Humidity'] + 1)
    df['Water_Balance'] = df['Rainfall_mm'] + df['Previous_Irrigation_mm'] - df['ET0']
    df['Moisture_Stress'] = (df['Temperature_C'] * (100 - df['Humidity'])) / (df['Soil_Moisture'] + 1)
    df['Crop_Growth'] = df['Crop_Type'] + '_' + df['Crop_Growth_Stage']
    df['Region_Season'] = df['Region'] + '_' + df['Season']
    return df

X = feature_engineering(X)
X_test = feature_engineering(X_test)

# ====================== 3. 分类特征处理 ======================
cat_features = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 
                'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region',
                'Crop_Growth', 'Region_Season']

for col in cat_features:
    X[col] = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

# ====================== 4. 类别权重 ======================
classes = np.unique(y)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y)
class_weights = dict(zip(classes, weights))
print(f"类别权重：{class_weights}")

# ====================== 5. 模型训练 ======================
skf = StratifiedKFold(n_splits=8, shuffle=True, random_state=42)
test_preds = np.zeros((len(X_test), 3))
val_f1_scores = []

print(f"\n开始训练（模式：{TRAINING_MODE}）...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n>>>>>>>>>>>>> 第 {fold+1} 折 <<<<<<<<<<<<<")
    
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    # 【最终修复版参数】根据模式自动调整
    model_params = {
        'iterations': 1000,
        'learning_rate': 0.1,
        'depth': 6,
        'cat_features': cat_features,
        'class_weights': class_weights,
        'eval_metric': 'TotalF1',
        'verbose': 50,
        'early_stopping_rounds': 200,
        'random_state': fold,
        'task_type': TRAINING_MODE
    }
    
    # CPU模式下可以加采样，GPU模式下不加（
    if TRAINING_MODE == 'CPU':
        model_params.update({
            'bootstrap_type': 'Bernoulli',
            'subsample': 0.8,
            'colsample_bylevel': 0.8
        })
    
    model = CatBoostClassifier(**model_params)
    
    # 训练
    model.fit(
        X_train_fold, y_train_fold,
        eval_set=(X_val_fold, y_val_fold),
        use_best_model=True
    )
    
    # 评估
    val_pred = model.predict(X_val_fold)
    f1 = f1_score(y_val_fold, val_pred, average='macro')
    val_f1_scores.append(f1)
    
    print(f"\n--- 第 {fold+1} 折验证报告 ---")
    print(classification_report(y_val_fold, val_pred, target_names=['Low', 'Medium', 'High']))
    
    # 预测
    test_preds += model.predict_proba(X_test) / skf.n_splits

print(f"\n>>>>> 平均 Macro-F1: {np.mean(val_f1_scores):.4f} <<<<<")

# ====================== 6. 生成提交文件 ======================
final_pred = np.argmax(test_preds, axis=1)
reverse_map = {0: "Low", 1: "Medium", 2: "High"}
submission['Irrigation_Need'] = final_pred
submission['Irrigation_Need'] = submission['Irrigation_Need'].map(reverse_map)

submission.to_csv(SUBMISSION_OUTPUT_PATH, index=False)
print(f"\n✅ 完成！文件已保存至：{SUBMISSION_OUTPUT_PATH}")

正在读取数据...
正在构造特征...
类别权重：{np.int64(0): np.float64(0.5676949153458749), np.int64(1): np.float64(0.8783891180136694), np.int64(2): np.float64(9.995716121662145)}

开始训练（模式：GPU）...

>>>>>>>>>>>>> 第 1 折 <<<<<<<<<<<<<
0:	learn: 0.9617682	test: 0.9622239	best: 0.9622239 (0)	total: 62.3ms	remaining: 1m 2s
50:	learn: 0.9654474	test: 0.9647543	best: 0.9650083 (48)	total: 2.16s	remaining: 40.3s
100:	learn: 0.9681562	test: 0.9664063	best: 0.9664063 (100)	total: 3.97s	remaining: 35.3s
150:	learn: 0.9700770	test: 0.9674292	best: 0.9677010 (132)	total: 5.87s	remaining: 33s
200:	learn: 0.9716097	test: 0.9682868	best: 0.9683136 (198)	total: 7.68s	remaining: 30.5s
250:	learn: 0.9729327	test: 0.9683374	best: 0.9686304 (208)	total: 9.53s	remaining: 28.4s
300:	learn: 0.9740491	test: 0.9686542	best: 0.9687697 (299)	total: 11.4s	remaining: 26.5s
350:	learn: 0.9749843	test: 0.9691978	best: 0.9693893 (336)	total: 13.3s	remaining: 24.5s
400:	learn: 0.9757824	test: 0.9693020	best: 0.9694329 (397)	total: 15.1s	re